
# Week 8 — RNA‑Seq with **Salmon** → **tximport** → **DESeq2** → **Figures**

This notebook is self‑contained for class and does:
1) Simulate a small **transcriptome** and paired‑end RNA‑seq reads (A vs B, 3 reps each)  
2) Trim with `fastp`  
3) **Quantify with Salmon** (quasi‑mapping to transcripts)  
4) Import with **tximport** and run **DESeq2** (gene‑level DE)  
5) Quick volcano + optional heatmap


## 0) Install fastp

In [ ]:
%%bash
set -e  # keep it simple; avoid -u -o pipefail causing spurious exits

# System deps
apt-get -qq update
apt-get -qq install -y fastp pigz wget || true

# Try apt salmon (if available in this VM)
if ! command -v salmon >/dev/null 2>&1; then
  apt-get -qq install -y salmon && echo "✅ Using apt salmon" || true
fi

# Fallback: download static Salmon, install under $HOME/tools
if ! command -v salmon >/dev/null 2>&1; then
  echo "⬇️  Installing Salmon from binary release (local user space)..."
  SALMON_VER="1.10.2"
  TGZ="salmon-${SALMON_VER}_linux_x86_64.tar.gz"
  URL="https://github.com/COMBINE-lab/salmon/releases/download/v${SALMON_VER}/${TGZ}"
  wget -qO "${TGZ}" "${URL}"
  tar -xzf "${TGZ}"
  S_DIR="salmon-${SALMON_VER}_linux_x86_64"
  mkdir -p "$HOME/tools/${S_DIR}"
  cp -r "${S_DIR}/"* "$HOME/tools/${S_DIR}/"
  chmod +x "$HOME/tools/${S_DIR}/bin/salmon"
  echo "$HOME/tools/${S_DIR}/bin" > /tmp/SALMON_DIR
  "$HOME/tools/${S_DIR}/bin/salmon" --version
  echo "✅ Salmon installed under $HOME/tools/${S_DIR}"
else
  # If apt installed salmon, record its bin dir (usually /usr/bin)
  dirname "$(command -v salmon)" > /tmp/SALMON_DIR
  salmon --version
fi



## 1) Build a tiny **transcriptome** and simulate reads
- 120 transcripts (~900 bp), one transcript per gene for simplicity  
- Group **B** has ~2× expression for 20 randomly chosen genes  
- Paired‑end 75bp reads, 20k pairs per replicate × 3 reps per group  
- Save `toy_transcripts.fa` and `tx2gene.tsv` for tximport


In [ ]:

import numpy as np, random, textwrap, csv
np.random.seed(42); random.seed(42)

TX_N = 120
TX_LEN = 900
READ_LEN = 75
REPS = 3
READS_PER_REP = 20000

DE_IDXS = set(np.random.choice(range(TX_N), 20, replace=False))

seqs = []
tx2gene_rows = []
for i in range(TX_N):
    tx_id = f"tx{i}"
    gene_id = f"gene{i}"
    tx_seq = ''.join(np.random.choice(list("ACGT"), TX_LEN))
    seqs.append((tx_id, tx_seq))
    tx2gene_rows.append([tx_id, gene_id])

with open("toy_transcripts.fa","w") as fh:
    for tid, seq in seqs:
        fh.write(f">{tid}\n")
        for line in textwrap.wrap(seq, 80):
            fh.write(line+"\n")

with open("tx2gene.tsv","w", newline="") as fh:
    w = csv.writer(fh, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar='\\')
    w.writerow(["TXNAME","GENEID"])
    for row in tx2gene_rows:
        w.writerow(row)

def revcomp(s: str) -> str:
    tr = str.maketrans("ACGT","TGCA")
    return s.translate(tr)[::-1]

def simulate_group(label: str, reps=3, reads_per_rep=20000, read_len=75):
    base = np.random.lognormal(mean=1.5, sigma=0.7, size=TX_N)
    if label.upper() == "B":
        base = base * np.array([2.0 if i in DE_IDXS else 1.0 for i in range(TX_N)])
    p = base / base.sum()

    made = []
    for r in range(reps):
        r1, r2 = [], []
        for n in range(reads_per_rep):
            i = np.random.choice(TX_N, p=p)
            tid, tseq = seqs[i]
            start = np.random.randint(0, TX_LEN - 2*read_len - 1)
            frag = tseq[start:start+2*read_len]
            read1 = frag[:read_len]
            read2 = revcomp(frag[read_len:])
            q = "I"*read_len
            rid = f"{label}_rep{r}_read{n}"
            r1 += [f"@{rid}\n{read1}\n+\n{q}\n"]
            r2 += [f"@{rid}\n{read2}\n+\n{q}\n"]
        out1, out2 = f"{label}_rep{r}_R1.fq", f"{label}_rep{r}_R2.fq"
        with open(out1,"w") as f: f.writelines(r1)
        with open(out2,"w") as f: f.writelines(r2)
        made.append((out1,out2))
    return made

grpA = simulate_group("A", REPS, READS_PER_REP, READ_LEN)
grpB = simulate_group("B", REPS, READS_PER_REP, READ_LEN)
print("✅ Simulated reads written for A and B.")


## 2) Trim reads with **fastp**

In [ ]:

%%bash
set -e
for r1 in *_R1.fq; do
  base=${r1%_R1.fq}
  fastp -i ${base}_R1.fq -I ${base}_R2.fq         -o ${base}_R1.trim.fq -O ${base}_R2.trim.fq         -q 20 -u 30 -n 5 -l 30         -h ${base}_fastp.html -j ${base}_fastp.json >/dev/null
done
echo "✅ fastp trimming complete."


## 3) **Salmon** index and quantification

In [ ]:

%%bash
set -e
salmon index -t toy_transcripts.fa -i toy_salmon_idx -k 31 >/dev/null

for base in A_rep0 A_rep1 A_rep2 B_rep0 B_rep1 B_rep2; do
  salmon quant -i toy_salmon_idx -l A     -1 ${base}_R1.trim.fq -2 ${base}_R2.trim.fq     -p 2 -o ${base}_quant >/dev/null
done
echo "Salmon quantification complete."


## 4) Convert to using R so we can implement DESeq2

*   install rpy2 to run R in Python notebook
*   install tools (DESeq2, tximport)
*   use **tximport** to get Salmon quant results formatted for DESeq2
*   run DESeq2


In [ ]:
!pip -q install rpy2
%load_ext rpy2.ipython


In [ ]:
%%R

# --- Package installation (only installs if missing) ---
# BiocManager is the standard way to install Bioconductor packages like DESeq2 and tximport.
if (!requireNamespace("BiocManager", quietly=TRUE)) {
  install.packages("BiocManager", repos="https://cloud.r-project.org")
}
if (!requireNamespace("DESeq2", quietly=TRUE)) {
  BiocManager::install("DESeq2", ask=FALSE)
}
if (!requireNamespace("tximport", quietly=TRUE)) {
  BiocManager::install("tximport", ask=FALSE)
}

# readr is a tidyverse package for fast, clean file import (used below to read .tsv)

if (!requireNamespace("readr", quietly=TRUE)) {
  install.packages("readr", repos="https://cloud.r-project.org")
}


# --- Load required libraries into the R environment ---

library(tximport); library(DESeq2); library(readr)


# --- Define the sample names and file paths ---
# Here we have 6 samples total: 3 replicates for condition A and 3 for condition B

samples <- c("A_rep0","A_rep1","A_rep2","B_rep0","B_rep1","B_rep2")

# Build the path to each sample’s quant.sf file inside its Salmon output directory
files <- file.path(paste0(samples, "_quant"), "quant.sf")


# Name each file by its sample ID for easier handling downstream
names(files) <- samples


# --- Read the transcript-to-gene mapping table ---
# This table (tx2gene.tsv) links transcript IDs to their corresponding gene IDs.
# tximport uses it to collapse transcript-level quantification into gene-level counts.
tx2gene <- read_tsv("tx2gene.tsv", show_col_types = FALSE)

# --- Import Salmon quantification results ---
# tximport reads the quant.sf files, normalizes for transcript length,
# and summarizes transcript-level TPMs/NumReads to gene-level counts.
txi <- tximport(files, type="salmon", tx2gene=tx2gene, ignoreTxVersion=TRUE)

# --- Create sample metadata (colData) ---
# Define the experimental condition (factor variable) for each sample
condition <- factor(c("A","A","A","B","B","B"))

# Combine into a data frame with rownames matching sample names
coldata <- data.frame(row.names=samples, condition=condition)

# --- Create the DESeq2 dataset ---
# This constructs a DESeqDataSet object from the imported gene-level counts
# and attaches the sample metadata for downstream modeling.
dds <- DESeqDataSetFromTximport(txi, colData=coldata, design=~condition)

# --- Set reference level and run DESeq2 analysis ---
# Relevel the condition factor so "A" is treated as the baseline (reference)
dds$condition <- relevel(dds$condition, ref="A")

# Perform the DESeq2 pipeline:
# normalization → dispersion estimation → negative binomial fitting → hypothesis testing
dds <- DESeq(dds)

# --- Extract and organize results ---
# Generate the results table contrasting condition B vs A
res <- results(dds, contrast=c("condition","B","A"))

# Order the results by adjusted p-value (FDR)
res <- res[order(res$padj), ]

# --- Save and summarize ---
# Write the results table to a CSV for downstream visualization (Python or R)
write.csv(as.data.frame(res), file="DESeq2_salmon_tximport_results.csv")

# Display a summary of differential expression statistics
summary(res)


## 5) Create a volcano plot

*   Color genes with >= 2-fold change AND FDR <0.05
*  +(green) or -(red)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- settings (tweak as you like) ---
CSV_PATH = "DESeq2_salmon_tximport_results.csv"  # from your DESeq2 step
LFC_CUT  = 1.0    # absolute log2 fold-change threshold
PADJ_CUT = 0.05   # adjusted p-value threshold (FDR)
TOP_N    = 12     # labels per side (up & down)

# --- load & prep ---
res = pd.read_csv(CSV_PATH, index_col=0)
# ensure needed cols exist
assert {"log2FoldChange","padj"}.issubset(res.columns)

res = res.copy()
# numeric + -log10(padj)
res["padj"] = pd.to_numeric(res["padj"], errors="coerce")
res["neglog10padj"] = -np.log10(res["padj"])
res.loc[~np.isfinite(res["neglog10padj"]), "neglog10padj"] = np.nan

# calls: Up (green), Down (red), NS (black)
res["call"] = "NS"
res.loc[(res["padj"] < PADJ_CUT) & (res["log2FoldChange"] >=  LFC_CUT), "call"] = "Up"
res.loc[(res["padj"] < PADJ_CUT) & (res["log2FoldChange"] <= -LFC_CUT), "call"] = "Down"

# choose labels: top N by significance on each side
up_lab   = res.query("call == 'Up'").nlargest(TOP_N, "neglog10padj")
down_lab = res.query("call == 'Down'").nlargest(TOP_N, "neglog10padj")
labels_df = pd.concat([up_lab, down_lab])

# --- plot ---
plt.figure(figsize=(7.5, 5.5))

# plot order: NS (black) under, then Down (red), then Up (green)
ns   = res["call"] == "NS"
down = res["call"] == "Down"
up   = res["call"] == "Up"

plt.scatter(res.loc[ns,   "log2FoldChange"], res.loc[ns,   "neglog10padj"],
            s=10, c="black", alpha=0.6, label="NS")
plt.scatter(res.loc[down, "log2FoldChange"], res.loc[down, "neglog10padj"],
            s=14, c="red",   alpha=0.8, label="Down (FDR<{:.2g}, LFC≤-{})".format(PADJ_CUT, LFC_CUT))
plt.scatter(res.loc[up,   "log2FoldChange"], res.loc[up,   "neglog10padj"],
            s=14, c="green", alpha=0.8, label="Up (FDR<{:.2g}, LFC≥+{})".format(PADJ_CUT, LFC_CUT))

# cut lines
plt.axvline(-LFC_CUT, linestyle="--", linewidth=1, color="gray")
plt.axvline(+LFC_CUT, linestyle="--", linewidth=1, color="gray")
plt.axhline(-np.log10(PADJ_CUT), linestyle="--", linewidth=1, color="gray")

# labels for top hits
for gene, row in labels_df.iterrows():
    if np.isfinite(row["neglog10padj"]):
        plt.annotate(gene,
                     (row["log2FoldChange"], row["neglog10padj"]),
                     xytext=(3, 3), textcoords="offset points",
                     fontsize=8, color="black")

plt.title("DESeq2 Volcano (Salmon → tximport)")
plt.xlabel("log2 Fold Change (B vs A)")
plt.ylabel("-log10 adjusted p-value")
plt.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.show()


## 6) Create a heat map

*   Blue (down) | Red (up)


In [ ]:
# --- Updated color scheme: blue (cold) -> white -> red (hot) ---

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list

# ------------------------------
# Switches & config
# ------------------------------
CLUSTER_ROWS  = True   # cluster genes
CLUSTER_COLS  = True   # cluster samples
LINKAGE_METHOD = "average"
DIST_METRIC    = "euclidean"

DESEQ_CSV   = "DESeq2_salmon_tximport_results.csv"
TX2GENE_TSV = "tx2gene.tsv"
SAMPLES     = ["A_rep0","A_rep1","A_rep2","B_rep0","B_rep1","B_rep2"]
QUANT_PATHS = {s: os.path.join(f"{s}_quant","quant.sf") for s in SAMPLES}
TOP_N       = 30
LABEL_ROWS  = True

# ------------------------------
# Load DE results; choose top genes
# ------------------------------
res = pd.read_csv(DESEQ_CSV, index_col=0)
res = res.dropna(subset=["padj"]).sort_values("padj")
top_genes = [g for g in res.index[:TOP_N]]

# ------------------------------
# Build gene-level TPM matrix
# ------------------------------
tx2gene = pd.read_csv(TX2GENE_TSV, sep="\t").rename(columns={"TXNAME":"transcript","GENEID":"gene"})

tpm_tables = {}
for sample, qpath in QUANT_PATHS.items():
    q = pd.read_csv(qpath, sep="\t")[["Name","TPM"]].rename(columns={"Name":"transcript"})
    g = q.merge(tx2gene, on="transcript", how="left").dropna(subset=["gene"])
    tpm_tables[sample] = g.groupby("gene", as_index=True)["TPM"].sum()

tpm = pd.concat(tpm_tables, axis=1)
tpm = tpm.reindex(index=[g for g in top_genes if g in tpm.index])

# log2(TPM+1) and row z-score
log_tpm = np.log2(tpm + 1.0)
mu = log_tpm.mean(axis=1)
sd = log_tpm.std(axis=1).replace(0, np.nan)
z = log_tpm.sub(mu, axis=0).div(sd, axis=0)

# ------------------------------
# Optional hierarchical clustering
# ------------------------------
row_order = list(range(z.shape[0]))
col_order = list(range(z.shape[1]))

zmat = z.fillna(0).values

if CLUSTER_ROWS and z.shape[0] > 1:
    row_link = linkage(pdist(zmat, metric=DIST_METRIC), method=LINKAGE_METHOD, optimal_ordering=True)
    row_order = leaves_list(row_link)
if CLUSTER_COLS and z.shape[1] > 1:
    col_link = linkage(pdist(zmat.T, metric=DIST_METRIC), method=LINKAGE_METHOD, optimal_ordering=True)
    col_order = leaves_list(col_link)

z_ord = z.iloc[row_order, col_order]
row_labels = z_ord.index.tolist()
col_labels = z_ord.columns.tolist()

# ------------------------------
# Plot: blue-white-red color scheme
# ------------------------------
left, bottom = 0.12, 0.12
heat_w, heat_h = 0.65, 0.65
dend_h = 0.18 if CLUSTER_COLS else 0.0
dend_w = 0.18 if CLUSTER_ROWS else 0.0
gap = 0.02

fig = plt.figure(figsize=(9, 8))

# Column dendrogram (top)
if CLUSTER_COLS and z.shape[1] > 1:
    ax_col = plt.axes([left + dend_w, bottom + heat_h + gap, heat_w, dend_h])
    dendrogram(col_link, no_labels=True, color_threshold=None)
    ax_col.set_xticks([]); ax_col.set_yticks([])

# Row dendrogram (left)
if CLUSTER_ROWS and z.shape[0] > 1:
    ax_row = plt.axes([left, bottom, dend_w, heat_h])
    dendrogram(row_link, orientation="right", no_labels=True, color_threshold=None)
    ax_row.set_xticks([]); ax_row.set_yticks([])

# Heatmap (coolwarm colormap)
ax_heat = plt.axes([left + dend_w, bottom, heat_w, heat_h])
im = ax_heat.imshow(z_ord.values, aspect="auto", interpolation="nearest", cmap="coolwarm")


ax_heat.set_xticks(range(len(col_labels)))
ax_heat.set_xticklabels(col_labels, rotation=45, ha="right", fontsize=9)
if LABEL_ROWS:
    ax_heat.set_yticks(range(len(row_labels)))
    ax_heat.set_yticklabels(row_labels, fontsize=8)
else:
    ax_heat.set_yticks([])

ax_heat.set_xlabel("Samples")
ax_heat.set_ylabel("Genes (top by FDR)")

# Colorbar
cax = plt.axes([left + dend_w + heat_w + gap, bottom, 0.02, heat_h])
cb = plt.colorbar(im, cax=cax)
cb.set_label("Row z-score of log2(TPM+1)")
cb.ax.tick_params(labelsize=8)

plt.show()
